# 身体活动促进政策分析

按单元顺序运行。首次使用可在安装单元将 `INSTALL_DEPENDENCIES` 设为 `True`；安装后重启内核并改回 `False`。


## 1 环境与数据


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/public_inputs.py").exists())
if str(ROOT / "src") not in sys.path: sys.path.insert(0, str(ROOT / "src"))
excel_file = ROOT / "data/Public_model_inputs.xlsx"


In [ ]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")])


读取工作簿并准备模型输入。输入表保留原有单位和来源字段。


In [ ]:
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
excel_tables = pd.read_excel(excel_file, sheet_name=None, dtype=object, keep_default_na=False)
from public_inputs import prepare_public_inputs
input_catalog = prepare_public_inputs(excel_tables, excel_file)
display(input_catalog)


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, importlib
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/joint_psa_v1_2.py").exists())
WORK = ROOT / "work"
DET = WORK / "deterministic"
PSA_INPUT = WORK / "psa_input"
OUT = ROOT / "outputs"
for p in [WORK, DET / "data", DET / "results", PSA_INPUT, OUT]: p.mkdir(parents=True, exist_ok=True)
os.environ.update(ALLDIS_DATA_ROOT=str(ROOT), ALLDIS_OUTPUT_ROOT=str(DET), ALLDIS_ANALYSIS_ROOT=str(DET), ALLDIS_PSA_INPUT_ROOT=str(PSA_INPUT))
if str(ROOT / "src") not in sys.path: sys.path.insert(0, str(ROOT / "src"))
from model_io import install as configure_io, plain_bytes
configure_io()
from public_inputs import ensure_inputs
ensure_inputs()
try:
    from IPython.display import display
except ImportError:
    display = print
def save_csv(frame, filename):
    path = OUT / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    return path


In [ ]:
import run_d1_d10_stage as det
det = importlib.reload(det)
inputs = det.load_inputs()
population = det.build_population(inputs)
pa = inputs["pa"]
assert len(pa) == 22
assert np.allclose(pa.p_inactive + pa.p_active, 1)
print("模型输入已就绪。")
display(inputs["lag"])
display(det.build_rr_interface(inputs, "COMMON_STROKE_PIF"))

In [ ]:
x = population[population.year.eq(2025)].merge(pa, on=["sex", "age_group"], validate="one_to_one")
rows = []
for sex, g in list(x.groupby("sex")) + [("Both", x)]:
    rows.append({"sex": sex, "population": g.population_model.sum(),
                 "model_inactive_percent": 100 * np.average(g.p_inactive, weights=g.population_model)})
baseline = pd.DataFrame(rows)
baseline["paper_inactive_percent"] = baseline.sex.map(dict(zip(pd.read_csv(ROOT/"reference/parameter_tables/S04_as_reported.csv").iloc[:,0].map({"女性":"Female","男性":"Male","总体":"Both"}), pd.read_csv(ROOT/"reference/parameter_tables/S04_as_reported.csv").iloc[:,2].astype(float))))
baseline["difference_percentage_points"] = baseline.model_inactive_percent-baseline.paper_inactive_percent
save_csv(baseline, "paper_baseline_discrepancy.csv")
save_csv(pa, "model_baseline_age_sex.csv")

## 2 确定性健康效应


In [ ]:
import run_d1_d10_stage as model
model = importlib.reload(model)
inputs = model.load_inputs()
population = model.build_population(inputs)
case = model.AnalysisCase("PRIMARY")
burden_projection, burden_slopes = model.project_gbd(inputs["gbd_burden"], "burden", population, inputs["wpp_history"], case.d3_disease)
case_projection, case_slopes = model.project_gbd(inputs["gbd_cases"], "cases", population, inputs["wpp_history"], case.d3_disease)
print("疾病负担投影行数", len(burden_projection), "病例投影行数", len(case_projection))

In [ ]:
pa_trajectory = model.build_pa_trajectories(inputs, case)
rr = model.build_rr_interface(inputs, case.stroke_mapping)
burden = model.calculate_health(burden_projection, pa_trajectory, rr, inputs["lag"], case.d1_lag)
cases = model.calculate_health(case_projection, pa_trajectory, rr, inputs["lag"], case.d1_lag)
htn = model.calculate_hypertension(inputs, population, pa_trajectory, rr, case.d1_lag)
assert np.allclose(pa_trajectory.inactive_policy + pa_trajectory.active_policy, 1)
assert (burden.avoided_value >= -1e-8).all()
print("健康明细", burden.shape, cases.shape, htn.shape)

In [ ]:
payer_cost = model.select_policy_cost(inputs, case.d9_maintenance, "payer")
societal_cost = model.select_policy_cost(inputs, case.d9_maintenance, "societal")
scenario_names = inputs["policy"].set_index("scenario_id").scenario_name_cn.to_dict()
summary, annual = model.summarize_horizons(model.annual_summary(burden), cases, htn, payer_cost, scenario_names)
d4 = model.build_d4_cost_table(inputs)
display(summary[summary.scenario_id.eq("S6")])
compact = {"01_population_2025_2050.csv":population,"02_PA_baseline_from_parent.csv":inputs["pa"],
 "03_RR_binary_interface.csv":rr,"05_primary_horizon_summary.csv":summary,"06_primary_annual_results.csv":annual,
 "13_D4_cost_interface.csv":d4,"19_policy_cost_payer_2025_2050.csv":payer_cost,"20_policy_cost_societal_2025_2050.csv":societal_cost}
for name, frame in compact.items(): frame.to_csv(DET / "results" / name, index=False, encoding="utf-8-sig")
details = {"GBD_burden_projection_2025_2050.csv.gz":burden_projection,"GBD_cases_projection_2025_2050.csv.gz":case_projection,
 "PA_trajectory_primary.csv.gz":pa_trajectory,"health_burden_detail_primary.csv.gz":burden,
 "health_cases_detail_primary.csv.gz":cases,"hypertension_detail_primary.csv.gz":htn}
for name, frame in details.items(): frame.to_csv(DET / "data" / name,index=False,encoding="utf-8-sig",compression="gzip")

## 3 概率敏感性分析


In [ ]:
import prepare_psa_inputs_v1_2 as prep
prep=importlib.reload(prep)
prep.main()
policy=pd.read_csv(PSA_INPUT/"policy_effect_parameters_v1.2.csv")
display(policy[["scenario_id","theta_mean","beta_alpha","beta_beta","coverage_mean","ramp_years","coverage_distribution"]])
save_csv(policy,"policy_psa_parameters.csv")
qc=pd.read_csv(PSA_INPUT/"00_PSA_INPUT_QC_binary_v2.0.csv")
assert qc.passed.all()

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import run_psa_plan_v1_2 as runner
runner=importlib.reload(runner)
WORKERS=int(os.environ.get("PA_WORKERS", "4"))
BATCH_ROOT=WORK/"psa_rerun/psa_formal"
BATCH_ROOT.mkdir(parents=True,exist_ok=True)
plan=pd.read_csv(PSA_INPUT/"PSA_batch_plan_binary_v2.0.csv")
plan=plan[plan.run_type.eq("FORMAL")].copy()
def content_hash(path):
    import gzip
    raw=plain_bytes(path)
    raw=gzip.decompress(raw) if path.suffix==".gz" else raw
    return hashlib.sha256(raw).hexdigest()
fingerprint_files = list((ROOT/"src").glob("*.py")) + list((ROOT/"data").glob("*.xlsx")) + list((ROOT/"data").glob("*.json")) + list((ROOT/"config").glob("*.csv"))
fingerprint_files += [p for p in PSA_INPUT.iterdir() if p.name.endswith((".csv",".csv.gz"))]
fingerprint_files += [DET/"results/01_population_2025_2050.csv",ROOT/"src/aggregate_inputs.py",ROOT/"input/policy_costs_long_term/policy_cost_2025_2050_D9_main_interim.csv"]
payload={str(p.relative_to(ROOT)):content_hash(p) for p in sorted(fingerprint_files)}
payload["numpy_version"]=np.__version__;payload["pandas_version"]=pd.__version__
signature=hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()
def run_or_resume(row):
    bid=int(row.batch_id); folder=BATCH_ROOT/f"batch_{bid:02d}"
    checkpoint=folder/"wrapper_checkpoint.json"
    files=[folder/"01_joint_PSA_draws.csv.gz",folder/"01B_draw_audit.csv.gz",folder/"run_manifest.json",folder/"00_QC.csv"]
    if checkpoint.exists() and all(p.exists() for p in files):
        old=json.loads(checkpoint.read_text())
        try:
            intact = old.get("hashes")=={p.name:content_hash(p) for p in files}
        except (OSError, EOFError):
            intact = False
        if old.get("signature")==signature and intact:
            return bid,"RESUMED_VERIFIED"
    bid, code, log=runner.run_one(row,BATCH_ROOT,True,"COMMON_STROKE_PIF")
    if code: raise RuntimeError(log)
    manifest=json.loads((folder/"run_manifest.json").read_text())
    assert manifest["draws_completed"]==500 and manifest["status"]=="PASS"
    checkpoint.write_text(json.dumps({"signature":signature,"hashes":{p.name:content_hash(p) for p in files}},indent=2))
    return bid,"RERUN"
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures=[pool.submit(run_or_resume,row) for _,row in plan.iterrows()]
    for future in as_completed(futures): print(future.result(),flush=True)

In [ ]:
parts=[]
for _, row in plan.iterrows():
    path = BATCH_ROOT/f"batch_{int(row.batch_id):02d}/01_joint_PSA_draws.csv.gz"
    try:
        part = pd.read_csv(path)
    except (OSError, EOFError):
        bid, code, log = runner.run_one(row, BATCH_ROOT, True, "COMMON_STROKE_PIF")
        if code: raise RuntimeError(log)
        part = pd.read_csv(path)
    parts.append(part)
draws=pd.concat(parts,ignore_index=True).sort_values(["draw_id","horizon_years","scenario_id"]).reset_index(drop=True)
if len(draws)!=240000 or draws.duplicated(["draw_id","horizon_years","scenario_id"]).any():
    raise ValueError("Incomplete formal run")
MERGED=WORK/"psa_rerun/merged";MERGED.mkdir(exist_ok=True)
draws.to_csv(MERGED/"01_joint_PSA_draws_merged.csv.gz",index=False,compression="gzip")
import merge_and_qc_psa_v1_2 as merge
summary=merge.summarize(draws)
summary.to_csv(MERGED/"02_PSA_summary.csv",index=False,encoding="utf-8-sig")
display(summary[summary.scenario_id.eq("S6")&summary.horizon_years.isin([5,10])])


## 4 成本效果与决策分析


In [ ]:
import merge_and_qc_psa_v1_2 as merge
import build_formal_postrun as post
draws=pd.read_csv(WORK/"psa_rerun/merged/01_joint_PSA_draws_merged.csv.gz")
summary=merge.summarize(draws)
ceac,ceaf=merge.build_ceac(draws,True)
frontier,incremental=post.build_frontier_and_incremental(draws,ceac)
evpi=post.build_evpi(draws)
for name,frame in {"PSA_summary.csv":summary,"multi_strategy_CEAC.csv":ceac,"CEAF.csv":ceaf,"frontier.csv":frontier,"incremental_ICER_NMB.csv":incremental,"EVPI.csv":evpi}.items():save_csv(frame,name)
display(frontier[frontier.perspective.eq("PROGRAMME_ONLY")&frontier.horizon_years.isin([5,10])])

In [ ]:
det=pd.read_csv(DET/"results/05_primary_horizon_summary.csv")
cols=["horizon_years","scenario_id","discounted_dalys_averted","discounted_programme_cost_2025_cny","icer_vs_s0_programme_only_cny_per_daly"]
actual=det[det.horizon_years.isin([5,10])&det.scenario_id.isin([f"S{i}" for i in range(1,7)])][cols]
save_csv(actual,"deterministic_CEA.csv")
cost=pd.read_csv(DET/"results/19_policy_cost_payer_2025_2050.csv")
cost["discounted_cost"]=cost.annual_programme_cost_2025_cny/1.03**(cost.year-2025)
display(actual)
save_csv(cost,"annual_policy_cost.csv")

In [ ]:
rows=[]
for h in [5,10]:
    d=draws[draws.horizon_years.eq(h)&draws.scenario_id.eq("S6")]
    for wtp in np.arange(0,200001,1000):
        probability=np.mean(wtp*d.discounted_dalys_averted-d.discounted_programme_cost_2025_cny>0)
        rows.append({"horizon_years":h,"wtp":wtp,"S6_probability":probability,"S0_probability":1-probability})
pairwise=pd.DataFrame(rows);save_csv(pairwise,"pairwise_S6_S0_CEAC.csv")
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(10,4),sharey=True)
for ax,(h,g) in zip(axes,pairwise.groupby("horizon_years")):
    ax.plot(g.wtp,g.S6_probability,label="S6");ax.plot(g.wtp,g.S0_probability,label="S0")
    ax.set(title=f"{h}-year",xlabel="WTP (CNY/DALY)",ylim=(0,1));ax.legend(frameon=False)
    ax.set_xticks([0,50000,100000,150000,200000], ["0","50,000","100,000","150,000","200,000"])
    ax.tick_params(axis="x",labelsize=9)
    for w in [50000,100000]:ax.axvline(w,color="grey",ls="--",lw=.7)
axes[0].set_ylabel("Probability cost-effective");fig.tight_layout()
fig.savefig(OUT/"CEAC_S6_vs_S0.png",dpi=250);fig.savefig(OUT/"CEAC_S6_vs_S0.pdf")
plt.show()

## 5 医疗费用与经济扩展


In [ ]:
import run_prepublication_extensions_v2 as econ
data=econ.load_inputs(ROOT,ROOT,DET,PSA_INPUT,WORK/"psa_rerun/merged/02_PSA_summary.csv")
params=econ.param_dict(data["macro_params"])
payer_share=econ.weighted_payer_share(params)
display(data["medical_params"])
display(data["macro_params"])
print("公共支付方份额",payer_share)

In [ ]:
four=econ.make_four_disease_annual(data)
four_summary=econ.summarise_four_diseases(four)
medical_detail,medical_annual,medical_summary=econ.build_medical_savings(data,four,payer_share)
display(medical_summary[medical_summary.scenario_id.eq("S6")])

In [ ]:
gdp=econ.gdp_mechanism(data,params)
roi_annual,roi_summary=econ.return_on_investment(data,medical_summary,medical_annual,gdp)
display(roi_summary[roi_summary.scenario_id.eq("S6")])

In [ ]:
uncertainty=econ.economic_satellite_uncertainty(data,medical_summary,gdp,params,n=10000,seed=20260803)
display(uncertainty[uncertainty.scenario_id.eq("S6")])
ECON=WORK/"economic";ECON.mkdir(exist_ok=True)
frames={"01_four_disease_annual_health.csv":four,"02_four_disease_horizon_health.csv":four_summary,
 "03_medical_savings_detail.csv":medical_detail,"04_medical_savings_annual.csv":medical_annual,
 "05_medical_savings_horizon.csv":medical_summary,"08_gdp_equivalent_mechanism_annual.csv":gdp,
 "10_return_on_investment_annual.csv":roi_annual,"11_return_on_investment_horizon.csv":roi_summary,
 "12_economic_satellite_uncertainty.csv":uncertainty}
for name,frame in frames.items():
    frame.to_csv(ECON/name,index=False,encoding="utf-8-sig")


## 6 疾病及人群分层


In [ ]:
burden=pd.read_csv(DET/"data/health_burden_detail_primary.csv.gz",low_memory=False)
cases=pd.read_csv(DET/"data/health_cases_detail_primary.csv.gz",low_memory=False)
htn=pd.read_csv(DET/"data/hypertension_detail_primary.csv.gz",low_memory=False)
rows=[]
for h in [5,10,26]:
    x=burden[burden.scenario_id.eq("S6")&burden.year.le(2024+h)].copy()
    x["discounted_value"]=x.avoided_value/1.03**(x.year-2025)
    q=x.groupby(["outcome_id","sex","age_group","measure_short"],as_index=False)["discounted_value"].sum();q["horizon_years"]=h;rows.append(q)
strata=pd.concat(rows,ignore_index=True)
save_csv(strata,"disease_age_sex_health.csv")
disease=strata.groupby(["horizon_years","outcome_id","measure_short"],as_index=False).discounted_value.sum()
save_csv(disease,"disease_DALY_YLL_YLD_deaths.csv")
quantity=[]
for h in [5,10,26]:
    c=cases[cases.scenario_id.eq("S6")&cases.year.le(2024+h)&cases.measure_short.eq("Incidence")].copy()
    t=htn[htn.scenario_id.eq("S6")&htn.year.le(2024+h)].copy()
    for frame,basis in [(c,"incident_events"),(t,"prevalence_equivalent_patient_years")]:
        z=frame.groupby(["outcome_id","sex","age_group"],as_index=False).avoided_value.sum();z["basis"]=basis;z["horizon_years"]=h;quantity.append(z)
quantity=pd.concat(quantity,ignore_index=True);save_csv(quantity,"disease_age_sex_quantity.csv")
display(disease[disease.horizon_years.eq(10)].pivot(index="outcome_id",columns="measure_short",values="discounted_value"))

In [ ]:
x=strata[strata.horizon_years.eq(10)]
shares=[]
for metric in ["DALY","Deaths"]:
    g=x[x.measure_short.eq(metric)];total=g.discounted_value.sum()
    for label,mask in [("age_60plus",g.age_group.isin(["60-64","65-69","70+"])),("age_70plus",g.age_group.eq("70+")),("Male",g.sex.eq("Male"))]:
        shares.append({"metric":metric,"group":label,"share":g.loc[mask,"discounted_value"].sum()/total})
shares=pd.DataFrame(shares);display(shares);save_csv(shares,"age_sex_contribution_shares.csv")
medical=pd.read_csv(WORK/"economic/03_medical_savings_detail.csv")
medical10=medical[medical.scenario_id.eq("S6")&medical.year.le(2034)&medical.cost_stream.eq("direct_medical")].groupby("outcome_id",as_index=False).discounted_savings_main_2025_cny.sum()
save_csv(medical10,"disease_medical_savings_10y.csv")

In [ ]:
import matplotlib.pyplot as plt
unit=medical[medical.scenario_id.eq("S6")&medical.cost_stream.eq("direct_medical")][["outcome_id","basis","unit_cost_main_2025_cny"]].drop_duplicates()
keys=["year","sex","age_group","outcome_id"]
frames=[]
for row in unit.itertuples(index=False):
    if row.outcome_id=="hypertension":
        z=htn[htn.scenario_id.eq("S6")&htn.year.le(2034)].copy()
    else:
        metric="Incidence" if row.basis=="avoided_incident_case_first_year" else "Prevalence"
        z=cases[cases.scenario_id.eq("S6")&cases.year.le(2034)&cases.outcome_id.eq(row.outcome_id)&cases.measure_short.eq(metric)].copy()
    z["medical_savings_2025_cny"]=z.avoided_value*row.unit_cost_main_2025_cny/1.03**(z.year-2025)
    frames.append(z[keys+["medical_savings_2025_cny"]])
medstrata=pd.concat(frames,ignore_index=True)
medagg=medstrata.groupby("outcome_id").medical_savings_2025_cny.sum().sort_index()
refmed=medical10.set_index("outcome_id").discounted_savings_main_2025_cny.sort_index()
assert np.allclose(medagg,refmed,rtol=1e-10,atol=1e-5)
save_csv(medstrata.groupby(["age_group","sex","outcome_id"],as_index=False).medical_savings_2025_cny.sum(),"age_sex_disease_medical_10y.csv")
annual_m=pd.read_csv(WORK/"economic/04_medical_savings_annual.csv")
annual_g=pd.read_csv(WORK/"economic/08_gdp_equivalent_mechanism_annual.csv")
m=annual_m[annual_m.scenario_id.eq("S6")].sort_values("year")
g=annual_g[annual_g.scenario_id.eq("S6")&annual_g.variant.eq("MAIN")].sort_values("year")
trajectory=pd.DataFrame({"year":m.year.to_numpy(),"annual_medical_bn":m.direct_medical_savings_main_2025_cny.to_numpy()/1.03**(m.year.to_numpy()-2025)/1e9,
                        "annual_gdp_equivalent_bn":g.discounted_gdp_equivalent_gain_2025_cny.to_numpy()/1e9})
trajectory["cumulative_medical_bn"]=trajectory.annual_medical_bn.cumsum()
trajectory["cumulative_gdp_equivalent_bn"]=trajectory.annual_gdp_equivalent_bn.cumsum()
save_csv(trajectory,"annual_cumulative_economic_trajectory.csv")


In [ ]:
from reporting import build_tables
build_tables(ROOT)

## 7 成本及参数表


In [ ]:
from reporting import build_cost_and_parameters
build_cost_and_parameters(ROOT)

## 8 图形与图中数据


In [ ]:
from figures import build_figures
build_figures(ROOT)

## 9 结果对应与核对


In [ ]:
from coverage_report import build_report
build_report(ROOT)